In [1]:
# Create a folder for the engine
!mkdir -p /kaggle/working/stockfish_engine

# Download the sf_19 Linux x86-64 universal binary
!wget -O /kaggle/working/stockfish.tar.gz https://github.com/official-stockfish/Stockfish/releases/download/sf_19/stockfish-linux-x86-64-universal.tar.gz

# Extract it
!tar -xvf /kaggle/working/stockfish.tar.gz -C /kaggle/working/stockfish_engine --strip-components=1

# Give execution permissions to the extracted binary
!chmod +x /kaggle/working/stockfish_engine/stockfish*

--2026-10-18 06:55:28--  https://github.com/official-stockfish/Stockfish/releases/download/sf_19/stockfish-linux-x86-64-universal.tar.gz
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/20976138/383849d9-c36e-42b9-8cc8-12852bcf97fc?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-10-18T07%3A31%3A47Z&rscd=attachment%3B+filename%3Dstockfish-linux-x86-64-universal.tar.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-10-18T06%3A30%3A55Z&ske=2026-10-18T07%3A31%3A47Z&sks=b&skv=2018-11-09&sig=G50BN2R929OJ%2F73YEN6cxjzKgTt3qeYXcb6%2ByODZIk4%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc5MjMwODMyOCwibmJmIjo

In [2]:
!pip install chess

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 79.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147879 sha256=bc8ac88a8c019910f556c692dda330928e6515873cd88487c8cd17f1a0f85a08
  Stored in directory: /root/.cache/pip/wheels/74/65/99/8c3a88ba852c1ed8d34daed144b8cb4c654d638677483d19ff
Successfully built chess


In [3]:
import chess
import chess.engine
import csv
import random
import statistics
import os

# --- KAGGLE CONFIGURATION ---
STOCKFISH_PATH = "/kaggle/working/stockfish_engine/stockfish-linux-x86-64-universal"
INPUT_FILE = "/kaggle/input/datasets/sslm11/1000000chesspositions/optimal_positions.txt"
OUTPUT_FILE = "/kaggle/working/continuous_pilot_results_d20.csv"

# Evaluate EVERY depth continuously from 1 to 20
DEPTHS = list(range(1, 21)) 
# Generate fractional epsilons from 0.0 to 1.0 in 0.05 increments for relative error
EPSILONS = [round(x * 0.05, 2) for x in range(21)]
PILOT_LIMIT = 100 

def run_kaggle_evaluations():
    print("Spinning up Stockfish on Kaggle...", flush=True)
    try:
        engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)
        # Give the single engine access to 4 cores and 16GB of Hash RAM (16384 MB)
        engine.configure({"Threads": 4, "Hash": 16384}) 
    except FileNotFoundError:
        print(f"Error: Stockfish not found at {STOCKFISH_PATH}.", flush=True)
        print("Please check that the tar.gz extracted successfully.", flush=True)
        return

    raw_data = {depth: {e: [] for e in EPSILONS} for depth in DEPTHS}

    print("Loading the massive FEN dataset into memory...", flush=True)
    try:
        with open(INPUT_FILE, "r") as fen_file:
            all_fens = [line.strip() for line in fen_file if line.strip()]
    except FileNotFoundError:
        print(f"Error: Could not find FEN file at {INPUT_FILE}.", flush=True)
        return
    
    sample_size = min(PILOT_LIMIT, len(all_fens))
    print(f"Randomly selecting {sample_size} positions...", flush=True)
    fens_to_evaluate = random.sample(all_fens, sample_size)

    print(f"\n" + "="*50, flush=True)
    print("STARTING CONTINUOUS STREAM (TARGET: D20 on 4 THREADS)", flush=True)
    print("="*50, flush=True)

    for i, fen in enumerate(fens_to_evaluate):
        board = chess.Board(fen)
        legal_moves = list(board.legal_moves)
        num_moves = len(legal_moves)

        if num_moves == 0:
            continue

        # Print EVERY position so you can spot exactly where it hangs
        if (i + 1) % 1 == 0:
            print(f"  Streaming Position {i+1}/{sample_size} (Depths 1->20)...", flush=True)

        # Track (is_mate, score_value)
        latest_known_scores = {move: (False, -10000) for move in legal_moves}
        depth_snapshots = {d: {} for d in DEPTHS}

        # STREAMING OPENED: Run to depth 20 exactly once
        with engine.analysis(board, chess.engine.Limit(depth=20), multipv=num_moves) as analysis:
            for info in analysis:
                d = info.get("depth")
                pv = info.get("pv")
                score_obj = info.get("score")

                if d is not None and pv is not None and score_obj is not None and len(pv) > 0:
                    if d <= 20:
                        root_move = pv[0]
                        pov_score = score_obj.pov(board.turn)
                        is_mate = pov_score.is_mate()
                        val = pov_score.score(mate_score=10000)
                        
                        latest_known_scores[root_move] = (is_mate, val)
                        depth_snapshots[d][root_move] = (is_mate, val)

        # Process the snapshots
        for d in DEPTHS:
            snapshot = depth_snapshots[d]
            for move in legal_moves:
                if move not in snapshot:
                    snapshot[move] = latest_known_scores[move]

            # Identify the best score in the current snapshot
            top_is_mate, top_score = max(snapshot.values(), key=lambda x: x[1])
            
            raw_losses = {}
            for move, (is_mate, score) in snapshot.items():
                # Both moves lead to winning mate -> relative error is 0
                if top_is_mate and top_score > 0 and is_mate and score > 0:
                    raw_losses[move] = 0
                else:
                    raw_losses[move] = top_score - score
            
            # Find the maximum loss to serve as the relative denominator for this specific state
            max_loss = max(raw_losses.values()) if raw_losses else 0
            
            epsilon_counts = {e: 0 for e in EPSILONS}

            for move, loss in raw_losses.items():
                # Normalize against the worst move in the position (state-relative error)
                rel_error = (loss / max_loss) if max_loss > 0 else 0.0
                
                for e in EPSILONS:
                    if rel_error <= e:
                        epsilon_counts[e] += 1

            for e in EPSILONS:
                rel_freq = epsilon_counts[e] / num_moves
                raw_data[d][e].append(rel_freq)

    engine.quit()

    print(f"\nCalculating Variance and saving data to {OUTPUT_FILE}...", flush=True)
    with open(OUTPUT_FILE, mode='w', newline='', encoding='utf-8') as csv_file:
        writer = csv.writer(csv_file)
        
        header = ["Depth", "Metric"] + [f"Rel_Error_<={e:.2f}" for e in EPSILONS]
        writer.writerow(header)
        
        for depth in DEPTHS:
            mean_row = [depth, "Mean"]
            stdev_row = [depth, "Std_Dev"]
            
            for e in EPSILONS:
                data_list = raw_data[depth][e]
                
                if len(data_list) > 1:
                    mean_row.append(round(statistics.mean(data_list), 4))
                    stdev_row.append(round(statistics.stdev(data_list), 4))
                elif len(data_list) == 1:
                    mean_row.append(round(data_list[0], 4))
                    stdev_row.append(0.0)
                else:
                    mean_row.append(0.0)
                    stdev_row.append(0.0)
                
            writer.writerow(mean_row)
            writer.writerow(stdev_row)

    print("Data extraction complete! You can now download the CSV from the Kaggle working directory.", flush=True)

if __name__ == "__main__":
    run_kaggle_evaluations()

Spinning up Stockfish on Kaggle...
Loading the massive FEN dataset into memory...
Randomly selecting 100 positions...

STARTING CONTINUOUS STREAM (TARGET: D20 on 4 THREADS)
  Streaming Position 1/100 (Depths 1->20)...
  Streaming Position 2/100 (Depths 1->20)...
  Streaming Position 3/100 (Depths 1->20)...
  Streaming Position 4/100 (Depths 1->20)...
  Streaming Position 5/100 (Depths 1->20)...
  Streaming Position 6/100 (Depths 1->20)...
  Streaming Position 7/100 (Depths 1->20)...
  Streaming Position 8/100 (Depths 1->20)...
  Streaming Position 9/100 (Depths 1->20)...
  Streaming Position 10/100 (Depths 1->20)...
  Streaming Position 11/100 (Depths 1->20)...
  Streaming Position 12/100 (Depths 1->20)...
  Streaming Position 13/100 (Depths 1->20)...
  Streaming Position 14/100 (Depths 1->20)...
  Streaming Position 15/100 (Depths 1->20)...
  Streaming Position 16/100 (Depths 1->20)...
  Streaming Position 17/100 (Depths 1->20)...
  Streaming Position 18/100 (Depths 1->20)...
  Stream